In [ ]:
!pip install polars catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os
import glob

# 1. Force Remount to see new shortcuts
print("🔄 Rafraîchissement du Drive...")
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

# 2. Search for the parquet files recursively
print("\n🔍 Recherche automatique des fichiers parquet.")

# Search for any file ending in .parquet inside MyDrive
found_files = glob.glob('/content/drive/MyDrive/**/part_00000.parquet', recursive=True)

if len(found_files) > 0:
    print(f"\n SUCCÈS ! Fichier trouvé ici :")
    print(f"{found_files[0]}")

    # Extract the correct base path
    full_path = found_files[0]
    directory = os.path.dirname(full_path) # /train_parquet
    base_path = os.path.dirname(directory) # /AMEX Challenge (or whatever it is named)

    print(f"\n Votre dossier racine correct est :")
    print(f"'{base_path}'")

    print("\n Copiez ce chemin pour la suite du code.")
else:
    print("\n introuvable.")


🔄 Rafraîchissement du Drive...
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive

🔍 Recherche automatique des fichiers parquet...

✅ SUCCÈS ! Fichier trouvé ici :
/content/drive/MyDrive/AMEX Challenge/train_parquet/part_00000.parquet

📂 Votre dossier racine correct est :
'/content/drive/MyDrive/AMEX Challenge'

👉 Copiez ce chemin pour la suite du code.


In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import glob
import gc
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# VOTRE CHEMIN (Vérifiez qu'il pointe bien vers le dossier contenant train_parquet, etc.)
base_path = '/content/drive/MyDrive/AMEX Challenge'


# Petite vérification immédiate pour éviter les frustrations
if glob.glob(f'{base_path}/train_parquet/*.parquet'):
    print(f" Chemin validé : {base_path}")
else:
    print(f" Erreur : Aucun fichier trouvé dans {base_path}/train_parquet")
    print("Vérifiez votre raccourci Google Drive.")

✅ Chemin validé : /content/drive/MyDrive/AMEX Challenge


In [ ]:
!pip install polars catboost

import polars as pl
import pandas as pd
import numpy as np
import glob
import gc
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import math
import os

# DEFINIR  CHEMIN CORRECT ICI
base_path = '/content/drive/MyDrive/AMEX Challenge'
def process_data_polars_with_lags(source):
    # Chargez les données (lazy)
    q = pl.scan_parquet(source)

    # 1. Trier par date (CRUCIAL pour que le "shift" ait du sens)
    q = q.sort(["customer_ID", "S_2"])

    # 2. Séparer les colonnes
    cat_features = ['B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68']
    try: schema = q.collect_schema()
    except: schema = q.schema
    num_features = [col for col in schema.names() if col not in cat_features + ['customer_ID', 'S_2']]

    # 3. Création des expressions d'agrégation
    exprs = []

    # --- A. Stats classiques ---
    for col in num_features:
        exprs.append(pl.col(col).mean().alias(f"{col}_mean"))
        exprs.append(pl.col(col).last().alias(f"{col}_last"))
        # Tendance (Last - Mean)
        exprs.append((pl.col(col).last() - pl.col(col).mean()).alias(f"{col}_trend"))

    for col in cat_features:
        exprs.append(pl.col(col).last().alias(f"{col}_last"))
        exprs.append(pl.col(col).n_unique().alias(f"{col}_count_unique"))

    # --- B. LAG FEATURES (Version Sécurisée avec shift) ---
    key_features = ['P_2', 'B_1', 'B_2', 'B_9', 'D_39', 'S_3']

    for col in key_features:
        # Lag 1 : On décale tout d'1 cran et on prend le dernier.
        # Si historique trop court -> renvoie null (pas d'erreur)
        exprs.append(pl.col(col).shift(1).last().alias(f"{col}_lag_1"))

        # Lag 2 : On décale de 2 crans
        exprs.append(pl.col(col).shift(2).last().alias(f"{col}_lag_2"))

        # Différence : Dernier - (Dernier décalé de 1)
        exprs.append((pl.col(col).last() - pl.col(col).shift(1).last()).alias(f"{col}_diff_last_lag1"))

    # Exécution
    return q.group_by("customer_ID").agg(exprs).collect().to_pandas()

In [ ]:
print("🏗️ --- ÉTAPE 1 : TRAITEMENT DU TRAIN ---")

# 1. Agrégation
print(" Lecture et agrégation du Train.")
train_files = f'{base_path}/train_parquet/*.parquet'
train_df = process_data_polars_with_lags(train_files)

# 2. Fusion des Labels
print(" Fusion des labels.")
train_labels = pd.read_csv(f'{base_path}/train_labels.csv')
train_df = train_df.merge(train_labels, on='customer_ID', how='left')

# 3. Sauvegarde intermédiaire (Checkpoint)
parquet_path = f'{base_path}/train_aggregated.parquet'
print(f" Sauvegarde de {parquet_path}.")
train_df.to_parquet(parquet_path)

# 4. VIDER LA RAM
del train_df, train_labels
gc.collect()
print(" RAM libérée. Train sauvegardé.")

🏗️ --- ÉTAPE 1 : TRAITEMENT DU TRAIN ---
⏳ Lecture et agrégation du Train...
⏳ Fusion des labels...
💾 Sauvegarde de /content/drive/MyDrive/AMEX Challenge/train_aggregated.parquet...
🧹 RAM libérée. Train sauvegardé.


In [ ]:
import math
import glob
import os
import gc

print(" --- ÉTAPE 2: GÉNÉRATION DU TEST AVEC LAGS --")

# 1. Lister tous les fichiers bruts
test_files = sorted(glob.glob(f'{base_path}/test_parquet/*.parquet'))
batch_size = 5
num_batches = math.ceil(len(test_files) / batch_size)

print(f" Mise à jour des {num_batches} lots de test avec les nouvelles features.")

for i in range(num_batches):
    save_name = f'{base_path}/test_batch_{i}.parquet'

    # ⚠️ IMPORTANT : On force l'écrasement des anciens fichiers
    if os.path.exists(save_name):
        print(f" Mise à jour du Lot {i+1} (écrasement de l'ancienne version).")
    else:
        print(f" Création du Lot {i+1}.")

    # Sélectionner le sous-groupe
    start = i * batch_size
    end = min((i + 1) * batch_size, len(test_files))
    current_batch_files = test_files[start:end]

    # On utilise la fonction avec les LAGS
    df_batch = process_data_polars_with_lags(current_batch_files)

    # Sauvegarder
    df_batch.to_parquet(save_name)

    # Vider RAM
    del df_batch
    gc.collect()

print("✅ Tous les lots de Test sont maintenant synchronisés avec le Train !")

🏗️ --- ÉTAPE 2 (CORRECTIF) : RE-GÉNÉRATION DU TEST AVEC LAGS ---
📦 Mise à jour des 5 lots de test avec les nouvelles features...
♻️  Mise à jour du Lot 1 (écrasement de l'ancienne version)...
♻️  Mise à jour du Lot 2 (écrasement de l'ancienne version)...
♻️  Mise à jour du Lot 3 (écrasement de l'ancienne version)...
♻️  Mise à jour du Lot 4 (écrasement de l'ancienne version)...
♻️  Mise à jour du Lot 5 (écrasement de l'ancienne version)...
✅ Tous les lots de Test sont maintenant synchronisés avec le Train !


In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
import gc

# CHARGEMENT
base_path = '/content/drive/MyDrive/AMEX Challenge'
print(" Chargement du Train complet.")
train_df = pd.read_parquet(f'{base_path}/train_aggregated.parquet')

# PREPARATION
features = [c for c in train_df.columns if c not in ['customer_ID', 'target', 'S_2']]
X = train_df[features]
y = train_df['target']

# PARTIE 1 : CRÉATION DE LA META-FEATURE (LGBM) ---
print(" Construction de la Meta-Feature (Stratégie du 5ème place).")

# Configuration K-Fold (5 plis)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models_lgb = []

# Paramètres légers pour la Meta-Feature (rapide)
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'n_jobs': -1,
    'verbose': -1,
    'random_state': 42
}

# Boucle d'entraînement K-Fold
for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y)):
    print(f"   🔹 Fold {fold+1}/5.")

    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val)

    model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=1000,
        valid_sets=[dtrain, dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)] # Silencieux
    )

    # On stocke la prédiction sur la partie VALIDATION (OOF)
    oof_preds[val_idx] = model.predict(X_val)
    models_lgb.append(model)

    del X_train, y_train, X_val, y_val, dtrain, dval
    gc.collect()

print(" Meta-feature générée pour le Train !")

# AJOUT DE LA META FEATURE AU DATASET
# La prédiction devient une colonne comme "age" ou "salaire"
X['meta_LGBM_pred'] = oof_preds

# --- PARTIE 2 : ENTRAÎNEMENT DU MODÈLE FINAL (CATBOOST) ---
print("\n Entraînement du modèle Principal (CatBoost) avec la Meta-Feature.")

import catboost as cb

# CatBoost va maintenant apprendre en utilisant la prédiction de LGBM comme aide
cat_model = cb.CatBoostClassifier(
    iterations=1500, # Augmenté car on a de meilleures features
    depth=6,
    learning_rate=0.05,
    eval_metric='Logloss',
    verbose=100,
    random_state=42,
    task_type="CPU" # Mettre "GPU" si Colab Pro
)

cat_model.fit(X, y)

print(" Modèle final entraîné.")

# Sauvegarde des modèles LGBM (pour le test) et Catboost
cat_model.save_model(f'{base_path}/catboost_meta.cbm')
# On garde les modèles LGB en mémoire dans la liste 'models_lgb' pour l'étape suivante

⏳ Chargement du Train complet...
🏗️ Construction de la Meta-Feature (Stratégie du 5ème place)...
   🔹 Fold 1/5...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[649]	training's binary_logloss: 0.156534	valid_1's binary_logloss: 0.2159
   🔹 Fold 2/5...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[504]	training's binary_logloss: 0.166658	valid_1's binary_logloss: 0.219535
   🔹 Fold 3/5...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[387]	training's binary_logloss: 0.176734	valid_1's binary_logloss: 0.219173
   🔹 Fold 4/5...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[406]	training's binary_logloss: 0.174964	valid_1's binary_logloss: 0.220119
   🔹 Fold 5/5...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[487]	training's binary_logloss:

/tmp/ipython-input-1671505833.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['meta_LGBM_pred'] = oof_preds


0:	learn: 0.6243556	total: 1.02s	remaining: 25m 31s
100:	learn: 0.2184359	total: 1m 41s	remaining: 23m 23s
200:	learn: 0.2169263	total: 3m 8s	remaining: 20m 20s
300:	learn: 0.2155948	total: 4m 36s	remaining: 18m 20s
400:	learn: 0.2142265	total: 5m 57s	remaining: 16m 20s
500:	learn: 0.2127216	total: 7m 26s	remaining: 14m 49s
600:	learn: 0.2112481	total: 8m 53s	remaining: 13m 18s
700:	learn: 0.2098378	total: 10m 19s	remaining: 11m 45s
800:	learn: 0.2084393	total: 11m 45s	remaining: 10m 15s
900:	learn: 0.2071013	total: 13m 10s	remaining: 8m 45s
1000:	learn: 0.2057704	total: 14m 37s	remaining: 7m 17s
1100:	learn: 0.2044711	total: 16m 4s	remaining: 5m 49s
1200:	learn: 0.2031731	total: 17m 33s	remaining: 4m 22s
1300:	learn: 0.2018595	total: 19m	remaining: 2m 54s
1400:	learn: 0.2006697	total: 20m 26s	remaining: 1m 26s
1499:	learn: 0.1994121	total: 21m 52s	remaining: 0us
✅ Modèle final entraîné.


In [ ]:
import glob
import os

print(" ÉTAPE 4 : PRÉDICTIONS AVEC META-FEATURES")

batch_files = sorted(glob.glob(f'{base_path}/test_batch_*.parquet'))
all_preds = []

# Poids pour l'ensemble final
# Ici on utilise LGB comme "helper" pour Catboost.
# Le résultat final est purement celui du Catboost qui contient déjà le savoir du LGB

for f in batch_files:
    print(f"Processing {os.path.basename(f)}.")
    df_chunk = pd.read_parquet(f)
    X_chunk = df_chunk[features] # Features originales uniquement

    # 1. GÉNÉRER LA META-FEATURE POUR CE BATCH
    # On utilise les 5 modèles LGBM entraînés précédemment et on fait la moyenne
    meta_preds = np.zeros(len(X_chunk))
    for model in models_lgb:
        meta_preds += model.predict(X_chunk) / len(models_lgb)

    # 2. AJOUTER LA COLONNE
    X_chunk['meta_LGBM_pred'] = meta_preds

    # 3. PRÉDICTION FINALE AVEC CATBOOST
    # Catboost voit maintenant les features originales + la prédiction moyenne de LGBM
    final_p = cat_model.predict_proba(X_chunk)[:, 1]

    mini_df = pd.DataFrame({
        'customer_ID': df_chunk['customer_ID'],
        'prediction': final_p
    })

    all_preds.append(mini_df)

    del df_chunk, X_chunk, mini_df, meta_preds
    gc.collect()

# Fusion et Sauvegarde
print("Assemblage final.")
submission = pd.concat(all_preds, axis=0)

# Gestion des doublons immédiate (plus besoin de script séparé)
submission = submission.groupby('customer_ID', as_index=False)['prediction'].mean()

save_path = f'{base_path}/submission_gold_strategy.csv'
submission.to_csv(save_path, index=False)
print(f" Fichier prêt : {save_path}")

🔮 --- ÉTAPE 4 : PRÉDICTIONS AVEC META-FEATURES ---
Processing test_batch_0.parquet...
Processing test_batch_1.parquet...
Processing test_batch_2.parquet...
Processing test_batch_3.parquet...
Processing test_batch_4.parquet...
🔗 Assemblage final...
🏆 Fichier prêt : /content/drive/MyDrive/AMEX Challenge/submission_gold_strategy.csv
